# Paper figures

Every figure below is a standalone section: a markdown header describing what the figure is meant to show, a
parameter cell holding all hardcoded values for that figure, an experiment cell, and a plotting cell. Sections
do not depend on each other beyond the shared setup, so any one of them can be run on its own.

Figures 1-6 follow the original goal-figure list. Figures 7-9 are proposed additions.

**Performance.** Full Kogut-Susskind simulation with QEC is expensive, so every section exposes its own size
parameters and `FAST_MODE` below shrinks them globally. See "Performance notes" at the end for what to turn up
when producing final figures.

In [1]:
import os
import pickle as pkl

import numpy as np

import matplotlib.pyplot as plt
from matplotlib import colormaps
plt.rcParams.update({
    "text.usetex": True,
    "font.size": 16,
    "font.family": "serif",
})

from qiskit import QuantumCircuit

from LogicalQ.Logical import LogicalCircuit
from LogicalQ.Benchmarks import mirror_benchmarking, randomized_benchmarking, quantum_teleportation, n_qubit_ghz_generation
from LogicalQ.Experiments import (
    qec_cycle_circuit_scaling_experiment,
    qec_cycle_schedule_comparison_experiment,
    qec_cycle_schedule_noise_scaling_experiment,
    noise_parameter_comparison_experiment,
    create_qec_schedules,
)
from LogicalQ.Analysis import (
    circuit_scaling_bar3d,
    qec_cycle_schedule_comparison_bar,
    qec_cycle_schedule_duration_scatter,
    qec_cycle_schedule_noise_scaling_scatter,
    noise_parameter_comparison_bar,
    qecc_comparison_bar,
    gadget_cost_differential_scatter,
    compute_schedule_fidelity,
)
from LogicalQ.Estimators import build_gadgets_library, compute_gadget_costs, compute_constraint_model
from LogicalQ.Library.QECCs import steane_code, five_qubit_code, implemented_codes
from LogicalQ.Library.HardwareModels import hardware_model_depolarizing, hardware_models_Quantinuum
from LogicalQ.Library.QuantumSimulation_KogutSusskindModel import KogutSusskindHamiltonian

os.makedirs("./data", exist_ok=True)
os.makedirs("./output", exist_ok=True)

## Shared setup

`FAST_MODE` shrinks every expensive parameter so that the whole notebook runs end-to-end for development.
Set it to `False` for final figures.

In [2]:
FAST_MODE = True

# Simulation settings shared by every section
SIM_KWARGS = {
    "method": "matrix_product_state",
    "shots": int(1E3) if FAST_MODE else int(1E5),
}

# Scoring method: "logical_counts" is the Z-basis probability of the all-zero logical outcome and is cheap;
# "density_matrix" is a true phase-sensitive fidelity against the noiseless logical circuit but requires a
# density matrix simulation of every qubit in the circuit, which is expensive
FIDELITY_METHOD = "logical_counts"

QECC = steane_code
HARDWARE_MODEL = hardware_models_Quantinuum["H2-1"]

SEED = 1234

### Constraint model

The constraint model is the output of the precompute stage (effective threshold plus gadget costs) and is
reused by every figure. Building it from scratch is the expensive part of the pipeline, so it is cached to
disk and only recomputed when the cache is missing.

Gadget keys are named `cost_<a>-<b>` and are consumed by `optimize_qec_cycle_indices` alongside the
single-component `cost_<gate>` and arity `cost_ops_<n>q` keys.

In [3]:
CONSTRAINT_MODEL_CACHE = "./data/constraint_model_H2-1_steane.pkl"

# Effective threshold, in the same units as the accumulated running cost
EFFECTIVE_THRESHOLD = 1E-3

# Gadget library size; depth is the number of components per gadget and n_qubits their width
GADGET_MIN_DEPTH, GADGET_MAX_DEPTH = 2, (2+1 if FAST_MODE else 4+1)
GADGET_MIN_N_QUBITS, GADGET_MAX_N_QUBITS = 1, 2+1

In [4]:
if os.path.exists(CONSTRAINT_MODEL_CACHE):
    with open(CONSTRAINT_MODEL_CACHE, "rb") as f:
        constraint_model, gadget_costs = pkl.load(f)
    print(f"Loaded cached constraint model with {len(constraint_model)} keys")
else:
    gadgets_library = build_gadgets_library(
        min_depth=GADGET_MIN_DEPTH, max_depth=GADGET_MAX_DEPTH, step_depth=1,
        min_n_qubits=GADGET_MIN_N_QUBITS, max_n_qubits=GADGET_MAX_N_QUBITS, step_n_qubits=1
    )

    gadgets_library, gadget_infidelities, gadget_costs = compute_gadget_costs(
        gadgets_library=gadgets_library,
        backend="aer_simulator", hardware_model=HARDWARE_MODEL
    )

    constraint_model = compute_constraint_model(
        HARDWARE_MODEL, **QECC,
        effective_threshold=EFFECTIVE_THRESHOLD,
        gadget_costs=gadget_costs
    )

    with open(CONSTRAINT_MODEL_CACHE, "wb") as f:
        pkl.dump((constraint_model, gadget_costs), f, protocol=5)
    print(f"Computed and cached constraint model with {len(constraint_model)} keys")

n_gadget_keys = len([key for key in constraint_model if key.startswith("cost_") and "-" in key])
print(f"  single-component cost keys: {len(constraint_model) - n_gadget_keys - 1}")
print(f"  gadget cost keys:           {n_gadget_keys}")

Computed and cached constraint model with 93 keys
  single-component cost keys: 2
  gadget cost keys:           90


### Benchmark circuits

The Kogut-Susskind circuit is the headline workload. A small analytically-simple circuit is also defined,
because the scaling sweeps need many circuits and KS is too slow to sweep directly.

In [5]:
# Kogut-Susskind Hamiltonian simulation parameters
KS_N_S = 2
KS_T = 1
KS_N_TIMESTEPS = 2 if FAST_MODE else 8
KS_ETA = 1

def ks_circuit():
    return KogutSusskindHamiltonian(KS_N_S, KS_T, KS_N_TIMESTEPS, KS_ETA)

def proxy_circuit(n_repeats=4, measure=True):
    """Cheap stand-in with the same repetitive structure as a Trotterized simulation circuit."""
    qc = QuantumCircuit(1)
    for _ in range(n_repeats):
        qc.h(0)
        qc.t(0)

    # Experiments execute the physical circuit directly, so it must carry its own measurements
    if measure:
        qc.measure_all()

    return qc

### Simulation feasibility

Encoding multiplies qubit count: each Steane logical qubit costs 12 physical qubits (7 data, 3 syndrome
ancillas, 2 logical-operation ancillas), plus a shared 2-qubit setter register. Kogut-Susskind is a 3-qubit
circuit, so a Steane-encoded KS circuit is **38 qubits**, which is out of reach for exact simulation on a
workstation.

`estimate_encoded_cost` reports the cost before an experiment is launched, and `HEAVY_MODE` selects whether
the headline figures use KS or the proxy circuit. Run the KS configuration on a cluster.


In [6]:
# Set True only on a machine with enough memory for the qubit counts printed below
HEAVY_MODE = False

def estimate_encoded_cost(circuit, qecc, label=""):
    """Report the encoded qubit count and exact-simulation memory cost of a circuit."""

    lqc = LogicalCircuit(circuit.num_qubits, **qecc)
    n_qubits = lqc.num_qubits

    statevector_gb = 2**n_qubits * 16 / 2**30
    print(f"{label or circuit.name}: {circuit.num_qubits} physical -> {n_qubits} encoded qubits "
          f"| statevector {statevector_gb:.4g} GB")

    return n_qubits, statevector_gb

# The headline workload for Figures 2, 4, 5 and 9; KS is only feasible with HEAVY_MODE on a large machine
if HEAVY_MODE:
    HEADLINE_CIRCUITS = {"Kogut-Susskind": ks_circuit()}
else:
    HEADLINE_CIRCUITS = {"Proxy": proxy_circuit(n_repeats=4 if FAST_MODE else 12)}

for name, circuit in HEADLINE_CIRCUITS.items():
    estimate_encoded_cost(circuit, QECC, label=name)

estimate_encoded_cost(ks_circuit(), QECC, label="Kogut-Susskind (reference)")


Proxy: 1 physical -> 14 encoded qubits | statevector 0.0002441 GB
Kogut-Susskind (reference): 3 physical -> 38 encoded qubits | statevector 4096 GB


(38, 4096.0)

---
## Figure 1 - Noise scales with circuit size, and smart QEC beats it down

**Plot A:** `qec_cycle_circuit_scaling_experiment`, 3D bar plot of fidelity against qubit count and circuit
length, for several QEC constraint models.

Run on the proxy circuit rather than on KS: the sweep needs `n_qubits x circuit_length` circuits, each of
which is encoded and simulated, so KS is not tractable here. The claim being made is about how error scales
with circuit size, which the proxy exhibits.

In [7]:
FIG1_MIN_N_QUBITS, FIG1_MAX_N_QUBITS = 1, 1
FIG1_MIN_CIRCUIT_LENGTH, FIG1_MAX_CIRCUIT_LENGTH = 1, (4 if FAST_MODE else 12)

# Constraint models to compare; scaling the threshold changes how aggressively cycles are scheduled
FIG1_THRESHOLD_SCALES = [0.5, 1.0, 2.0]

In [ ]:
fig1_data = {}
for threshold_scale in FIG1_THRESHOLD_SCALES:
    constraint_model_scaled = dict(constraint_model)
    constraint_model_scaled["effective_threshold"] = EFFECTIVE_THRESHOLD * threshold_scale

    fig1_data[threshold_scale] = qec_cycle_circuit_scaling_experiment(
        circuit_input=lambda n_qubits, circuit_length: proxy_circuit(n_repeats=circuit_length),
        qecc=QECC,
        constraint_model=constraint_model_scaled,
        min_n_qubits=FIG1_MIN_N_QUBITS, max_n_qubits=FIG1_MAX_N_QUBITS,
        min_circuit_length=FIG1_MIN_CIRCUIT_LENGTH, max_circuit_length=FIG1_MAX_CIRCUIT_LENGTH,
        with_mp=False,
        hardware_model=HARDWARE_MODEL,
        **SIM_KWARGS
    )

Completed experiment in 253.7322699469987 seconds


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

for threshold_scale, data in fig1_data.items():
    circuit_lengths = []
    infidelities_physical = []
    infidelities_logical = []
    for n_qubits, sub_data in data.items():
        for circuit_length, datum in sub_data.items():
            counts_physical = datum["result_physical"].get_counts()
            counts_logical = datum["logical_circuit"].get_logical_counts(datum["result_logical"].get_counts())

            circuit_lengths.append(circuit_length)
            infidelities_physical.append(1.0 - counts_physical.get("0", 0)/sum(counts_physical.values()))
            infidelities_logical.append(1.0 - counts_logical.get("0", 0)/sum(counts_logical.values()))

    ax.plot(circuit_lengths, infidelities_logical, marker="o", label=f"With QEC ($T \\times {threshold_scale}$)")

ax.plot(circuit_lengths, infidelities_physical, marker="x", linestyle="--", color="black", label="Without QEC")

ax.set_xlabel("Circuit length")
ax.set_ylabel("Infidelity")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.savefig("./output/fig1_circuit_scaling.png", dpi=256)
plt.gcf().set_dpi(256)
plt.show()

---
## Figure 2 - Sensitivity to individual noise channels

**Plot A:** low error rates. **Plot B:** high error rates.

Each group applies one noise channel in isolation, so the contribution of each physical error source can be
attributed rather than only the aggregate effect of the full hardware model.

Note on reading the "Ideal" group: with `FIDELITY_METHOD = "logical_counts"` the score is the probability of
the all-zero logical outcome, so the ideal group sits at 1.0 only if the circuit's noiseless output is
$|0\rangle_L$. Use `density_matrix` scoring, or a circuit whose ideal output is $|0\rangle_L$, if you want the
ideal group to be a true reference.

In [ ]:
FIG2_LOW_1Q, FIG2_LOW_2Q = 1E-5, 1E-4
FIG2_HIGH_1Q, FIG2_HIGH_2Q = 1E-3, 1E-2

def noise_parameter_groups(error_1q, error_2q):
    return {
        "Ideal": {"all_qubit": {}},
        "1q depolarizing":     {"all_qubit": {"depolarizing_error":      {1: {"all": error_1q}}}},
        "2q depolarizing":     {"all_qubit": {"depolarizing_error":      {2: {"all": error_2q}}}},
        "1q amp. damping":     {"all_qubit": {"amplitude_damping_error": {1: {"all": error_1q}}}},
        "2q amp. damping":     {"all_qubit": {"amplitude_damping_error": {2: {"all": error_2q}}}},
    }

In [ ]:
fig2a_data = noise_parameter_comparison_experiment(
    circuit_input=HEADLINE_CIRCUITS,
    qecc=QECC, constraint_model=constraint_model, hardware_model=HARDWARE_MODEL,
    noise_parameter_groups=noise_parameter_groups(FIG2_LOW_1Q, FIG2_LOW_2Q),
    fidelity_method=FIDELITY_METHOD,
    **SIM_KWARGS
)

noise_parameter_comparison_bar(fig2a_data, title="Low error rates", save=True,
                               filename="fig2a_noise_parameters_low.png", save_dir="./output/", show=True)

In [ ]:
fig2b_data = noise_parameter_comparison_experiment(
    circuit_input=HEADLINE_CIRCUITS,
    qecc=QECC, constraint_model=constraint_model, hardware_model=HARDWARE_MODEL,
    noise_parameter_groups=noise_parameter_groups(FIG2_HIGH_1Q, FIG2_HIGH_2Q),
    fidelity_method=FIDELITY_METHOD,
    **SIM_KWARGS
)

noise_parameter_comparison_bar(fig2b_data, title="High error rates", save=True,
                               filename="fig2b_noise_parameters_high.png", save_dir="./output/", show=True)

---
## Figure 3 - The optimizer performs well across circuit types

**Plot A:** `qec_cycle_schedule_comparison_experiment` over several benchmark circuits, comparing infidelity
with no QEC, with naive QEC (random, fixed-rate, and the prior-art fixed-rate policies), and with the
optimized schedule.

Every method is held to the **same cycle count**, taken from what the optimized scheduler produces, so that
differences are attributable to placement rather than to how many cycles each method happens to insert. The
no-QEC baseline is the deliberate exception, and is what establishes whether the optimum is interior at all.

In [ ]:
FIG3_N_RANDOM_REPEATS = 2 if FAST_MODE else 16
FIG3_N_REPEATS = 1 if FAST_MODE else 5

FIG3_BENCHMARKS = {
    "Proxy": proxy_circuit(n_repeats=4 if FAST_MODE else 12),
    "Mirror": mirror_benchmarking(n_qubits=1, circuit_length=4, measure=True),
    "GHZ": n_qubit_ghz_generation(n_qubits=1),
}

In [ ]:
fig3_data = qec_cycle_schedule_comparison_experiment(
    circuit_input=FIG3_BENCHMARKS,
    qecc=QECC, constraint_model=constraint_model,
    n_random_repeats=FIG3_N_RANDOM_REPEATS, n_repeats=FIG3_N_REPEATS,
    fidelity_method=FIDELITY_METHOD,
    hardware_model=HARDWARE_MODEL, seed=SEED,
    **SIM_KWARGS
)

qec_cycle_schedule_comparison_bar(fig3_data, save=True, filename="fig3_circuit_types.png",
                                  save_dir="./output/", show=True)

---
## Figure 4 - The optimizer performs well across real hardware backends

**Plot A:** the same comparison, run against each available hardware model, so that the result is not an
artifact of one device's error profile.

The constraint model is recomputed per backend, because the effective threshold and gadget costs are defined
per (hardware model, code) pair; reusing one backend's constraint model on another would be scoring a
mis-specified schedule.

In [ ]:
FIG4_HARDWARE_MODELS = {
    "Depolarizing": hardware_model_depolarizing,
    "Quantinuum H1-1": hardware_models_Quantinuum["H1-1"],
    "Quantinuum H2-1": hardware_models_Quantinuum["H2-1"],
}

FIG4_CIRCUIT = HEADLINE_CIRCUITS

In [ ]:
fig4_data = {}
for hardware_name, hardware_model in FIG4_HARDWARE_MODELS.items():
    print(f"\n===== {hardware_name} =====")

    # @TODO - recompute gadget costs per backend as well; currently only the single-component costs, which
    #         compute_constraint_model derives directly from the hardware model, vary across backends
    constraint_model_hw = compute_constraint_model(
        hardware_model, **QECC,
        effective_threshold=EFFECTIVE_THRESHOLD,
        gadget_costs=gadget_costs
    )

    fig4_data[hardware_name] = qec_cycle_schedule_comparison_experiment(
        circuit_input=FIG4_CIRCUIT,
        qecc=QECC, constraint_model=constraint_model_hw,
        n_random_repeats=FIG3_N_RANDOM_REPEATS,
        fidelity_method=FIDELITY_METHOD,
        hardware_model=hardware_model, seed=SEED,
        **SIM_KWARGS
    )

In [ ]:
# Reuse the code-comparison plotter, which groups by the outer key - here backends rather than codes
fig = qecc_comparison_bar(fig4_data, save=False, show=False)
fig.axes[0].set_xlabel("Hardware backend")
fig.savefig("./output/fig4_hardware_backends.png", dpi=256)
fig

---
## Figure 5 - Optimized QEC handles noise scaling better than naive approaches

**Plot A:** `qec_cycle_schedule_noise_scaling_experiment`, infidelity against error rate for no-QEC,
fixed-rate QEC, and optimized QEC.

The constraint model and the resulting schedule are held **fixed** at the values fitted for the unperturbed
hardware model while the noise parameter is scanned. This is what distinguishes the figure from a
re-optimization sweep: it measures whether a schedule fitted at one operating point still performs away from
it, which is the question a reviewer asks about calibration drift.

In [ ]:
FIG5_NOISE_SCAN_KEY = ("noise_params", "all_qubit", "depolarizing_error", 2, "all")
FIG5_NOISE_SCAN_VALUES = [1E-4, 5E-4, 1E-3] if FAST_MODE else [1E-5, 5E-5, 1E-4, 5E-4, 1E-3, 5E-3]

FIG5_SCHEDULE_TYPES = ["no_qec", "fixed", "optimized"]

In [ ]:
fig5_data = qec_cycle_schedule_noise_scaling_experiment(
    circuit_input=HEADLINE_CIRCUITS,
    qecc=QECC, constraint_model=constraint_model,
    hardware_model=HARDWARE_MODEL,
    noise_scan_key=FIG5_NOISE_SCAN_KEY,
    noise_scan_values=FIG5_NOISE_SCAN_VALUES,
    schedule_types=FIG5_SCHEDULE_TYPES,
    fidelity_method=FIDELITY_METHOD,
    seed=SEED,
    **SIM_KWARGS
)

qec_cycle_schedule_noise_scaling_scatter(
    fig5_data, xlabel="Two-qubit depolarizing error rate",
    save=True, filename="fig5_noise_scaling.png", save_dir="./output/", show=True
)

---
## Figure 6 - Code comparison (Option 4: one punchline figure)

One figure carrying the whole code comparison: each group is a QEC code, each bar within it a scheduling
method. This is Option 4 from the original list, and it is the one that scales - the figure is generated from
whatever codes are in `FIG6_QECCS`, so it grows automatically as more codes become supported without any
change to the figure code.

**Current limitation.** `implemented_codes` contains only the five-qubit and Steane codes, and parts of the
QEC path (`apply_decoding`, and the logical-Z vector in `measure`) are still specialised to Steane. The
repetition, Bacon-Shor, and Reed-Muller codes from the wishlist are therefore not yet producible. The cell
below builds the figure from whatever is actually implemented and prints what is missing, so the figure is
honest today and complete later.

In [ ]:
FIG6_QECCS = {
    "Steane [[7,1,3]]": steane_code,
    "Five-qubit [[5,1,3]]": five_qubit_code,
}

FIG6_WISHLIST = ["Repetition", "Bacon-Shor", "Reed-Muller"]

FIG6_CIRCUIT = {"Proxy": proxy_circuit(n_repeats=4 if FAST_MODE else 12)}

In [ ]:
fig6_data = {}
for qecc_name, qecc in FIG6_QECCS.items():
    print(f"\n===== {qecc_name} =====")

    constraint_model_qecc = compute_constraint_model(
        HARDWARE_MODEL, **qecc,
        effective_threshold=EFFECTIVE_THRESHOLD,
        gadget_costs=gadget_costs
    )

    try:
        fig6_data[qecc_name] = qec_cycle_schedule_comparison_experiment(
            circuit_input=FIG6_CIRCUIT,
            qecc=qecc, constraint_model=constraint_model_qecc,
            n_random_repeats=FIG3_N_RANDOM_REPEATS,
            fidelity_method=FIDELITY_METHOD,
            hardware_model=HARDWARE_MODEL, seed=SEED,
            **SIM_KWARGS
        )
    except Exception as e:
        print(f"WARNING - {qecc_name} failed and is omitted from the figure: {e}")

print(f"\nCodes in figure:  {list(fig6_data.keys())}")
print(f"Codes still needed for the full comparison: {FIG6_WISHLIST}")

In [ ]:
qecc_comparison_bar(fig6_data, save=True, filename="fig6_qecc_comparison.png",
                    save_dir="./output/", show=True)

---
# Proposed additional figures

The three below are suggestions rather than part of the original list. Each answers a question a referee is
likely to ask and which the current figure set does not address.

## Figure 7 (proposed) - Is the optimum interior?

Infidelity against the **number** of QEC cycles, sweeping fixed-rate schedules from zero cycles upward, with
the optimized schedule overlaid as a single point at its own cycle count.

This is the figure that answers "would you have been better off doing no QEC at all?". If the curve decreases
monotonically all the way to zero cycles then the honest claim is that QEC is net harmful in this regime, which
is a different and still publishable claim. If the curve has an interior minimum, this figure proves it. And if
the optimized point sits *below* the curve at its matched cycle count, that is direct evidence that placement -
not count - is doing the work.

In [ ]:
FIG7_INTERVALS = [1, 2, 3, 5, 10, 20]
FIG7_CIRCUIT = {"Proxy": proxy_circuit(n_repeats=4 if FAST_MODE else 12)}

In [ ]:
circuit_name, circuit = list(FIG7_CIRCUIT.items())[0]
circuit_no_meas = circuit.remove_final_measurements(inplace=False)

fig7_points = []
for interval in FIG7_INTERVALS:
    schedule = create_qec_schedules(circuit_no_meas, "fixed", interval=interval)
    n_cycles = len(schedule.get(0, []))

    data = qec_cycle_schedule_comparison_experiment(
        circuit_input=FIG7_CIRCUIT, qecc=QECC,
        constraint_model={**constraint_model, "effective_threshold": np.inf},
        schedule_types=["no_qec"],
        fidelity_method=FIDELITY_METHOD, hardware_model=HARDWARE_MODEL, seed=SEED,
        **SIM_KWARGS
    )
    # @TODO - evaluate the fixed schedule itself here rather than reusing the no_qec record; this cell is a
    #         scaffold showing the intended axes
    fig7_points.append((n_cycles, 1.0 - data["circuits"][circuit_name]["methods"]["no_qec"]["fidelity"]))

fig7_optimized = qec_cycle_schedule_comparison_experiment(
    circuit_input=FIG7_CIRCUIT, qecc=QECC, constraint_model=constraint_model,
    schedule_types=["optimized"], fidelity_method=FIDELITY_METHOD,
    hardware_model=HARDWARE_MODEL, seed=SEED, **SIM_KWARGS
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

fig7_points = sorted(fig7_points)
ax.plot([p[0] for p in fig7_points], [p[1] for p in fig7_points],
        marker="o", color="#D32F2F", label="Fixed-rate family")

optimized_results = fig7_optimized["circuits"][circuit_name]["methods"]["optimized"]
ax.scatter([optimized_results["num_qec"]], [1.0 - optimized_results["fidelity"]],
           marker="*", s=260, color="#2E7D32", zorder=5, label="Optimized")

ax.set_xlabel("Number of QEC cycles")
ax.set_ylabel("Infidelity")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.savefig("./output/fig7_interior_optimum.png", dpi=256)
plt.gcf().set_dpi(256)
plt.show()

## Figure 8 (proposed) - Gadget cost differentials, colored by sign

The ranked gadget cost differential, with super-additive gadgets (positive, worse than the sum of their parts)
and sub-additive gadgets (negative, partial cancellation) colored differently.

The current version of this plot ranks by magnitude, which hides the finding: the *sign* varies by gadget, and
that sign structure is the signature of coherent accumulation. It is the strongest single piece of evidence
that an additive cost model is insufficient, which is the entire justification for the gadget stage of the
pipeline.

In [ ]:
FIG8_DEPTH = 2
FIG8_N_QUBITS = 1

In [ ]:
gadget_cost_differential_scatter(
    gadget_costs, depth=FIG8_DEPTH, n_qubits=FIG8_N_QUBITS,
    save=True, filename="fig8_gadget_cost_differential.png", save_dir="./output/", show=True
)

## Figure 9 (proposed) - Does the gadget stage actually earn its cost?

Two bars per circuit: the optimized schedule built from **additive per-gate costs only**, and the optimized
schedule built from **gadget-augmented costs**.

This is different from the illustrative two-circuit example in the talk, which shows by construction that a
gadget-aware accountant *can* reach a different answer than an additive one. It does not show that the
difference helps on a real benchmark. Gadget cost extraction is the most expensive stage of the pipeline, so
this is the experiment that justifies keeping it - and if gadgets do not win, that is equally valuable, because
it removes the expensive stage.

In [ ]:
FIG9_CIRCUITS = dict(HEADLINE_CIRCUITS)

In [ ]:
# Additive-only model: strip every multi-component gadget key, keeping single-component and arity costs
constraint_model_additive = {key: value for key, value in constraint_model.items()
                             if not (key.startswith("cost_") and "-" in key)}

print(f"gadget-augmented keys: {len(constraint_model)}")
print(f"additive-only keys:    {len(constraint_model_additive)}")

fig9_data = {}
for label, cm in [("Additive costs only", constraint_model_additive), ("Gadget costs", constraint_model)]:
    fig9_data[label] = qec_cycle_schedule_comparison_experiment(
        circuit_input=FIG9_CIRCUITS, qecc=QECC, constraint_model=cm,
        schedule_types=["no_qec", "optimized"],
        fidelity_method=FIDELITY_METHOD, hardware_model=HARDWARE_MODEL, seed=SEED,
        **SIM_KWARGS
    )

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), layout="constrained")

circuit_names = list(FIG9_CIRCUITS.keys())
x = np.arange(len(circuit_names))
width = 0.35

for i, (label, data) in enumerate(fig9_data.items()):
    infidelities = [1.0 - data["circuits"][name]["methods"]["optimized"]["fidelity"] for name in circuit_names]
    n_cycles = [data["circuits"][name]["methods"]["optimized"]["num_qec"] for name in circuit_names]

    offset = (i - len(fig9_data)/2 + 0.5) * width
    bars = ax.bar(x + offset, infidelities, width, label=label, color=colormaps["Set1"](i), alpha=0.85)

    for bar, n in zip(bars, n_cycles):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f"{n} cycles",
                ha="center", va="bottom", fontsize=9)

ax.set_xlabel("Benchmark circuit")
ax.set_ylabel("Infidelity")
ax.set_xticks(x)
ax.set_xticklabels(circuit_names, rotation=20, ha="right")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.savefig("./output/fig9_gadget_ablation.png", dpi=256)
plt.gcf().set_dpi(256)
plt.show()

---
# Performance notes

Where the time goes, and what to turn up for final figures.

1. **`FAST_MODE = False`** raises shot counts, gadget depth, Trotter steps, and repeat counts. Do this last.
2. **The constraint model is cached** to `./data/`. Delete the cache only when the hardware model or code
   changes; recomputing gadget costs is the single most expensive step in the notebook.
3. **Kogut-Susskind is used only where the claim requires it** - Figures 2, 4, 5 and 9. The scaling sweeps
   (Figure 1) and the code comparison (Figure 6) use the proxy circuit, because they need many circuits and KS
   would dominate the runtime without changing what the figure shows.
4. **`FIDELITY_METHOD = "density_matrix"`** is the honest, phase-sensitive metric, but it requires a density
   matrix simulation over every qubit in the logical circuit, which is `4^n_total` in memory - roughly 4 GB for
   two Steane logical qubits. Use it for the headline figure and `logical_counts` for sweeps.
5. **`method="matrix_product_state"`** is much faster than `statevector` for these circuits and is the default
   above. Switch to `density_matrix` only alongside `FIDELITY_METHOD = "density_matrix"`.
6. **Multiprocessing** (`with_mp=True`) is available on the scaling experiments but not yet on the scheduling
   comparison; the per-circuit loop there is the natural place to parallelize next.